#Generación de Data sintética (Banking)

###Configuración

In [0]:
# Importar librerías
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
storage_account = "adlsprojectbank12"

base_path = (
    f"abfss://raw@{storage_account}.dfs.core.windows.net/banking"
)

# Volúmenes de prueba
N_CUSTOMERS = 1000
N_ACCOUNTS = 1500
N_CARDS = 1000
N_TRANSACTIONS = 50000
N_LOANS = 500
N_PAYMENTS = 5000

print("Configuración de generación")
print(f"Customers     : {N_CUSTOMERS:,}")
print(f"Accounts      : {N_ACCOUNTS:,}")
print(f"Cards         : {N_CARDS:,}")
print(f"Transactions  : {N_TRANSACTIONS:,}")
print(f"Loans         : {N_LOANS:,}")
print(f"Payments      : {N_PAYMENTS:,}")
print(f"Ruta base     : {base_path}")

###Generar customers

In [0]:
customers = (
    spark.range(1, N_CUSTOMERS + 1)
    .select(
        F.col("id").cast("long").alias("customer_id"),

        F.when(F.pmod("id", 2) == 0, "DNI")
         .otherwise("CE").alias("document_type"),

        F.lpad(F.col("id").cast("string"), 12, "0")
         .alias("document_number"),

        F.element_at(
            F.array(
                F.lit("Carlos"), F.lit("Maria"),
                F.lit("Jose"), F.lit("Ana"),
                F.lit("Luis"), F.lit("Lucia"),
                F.lit("Pedro"), F.lit("Rosa")
            ),
            (F.pmod("id", 8) + 1).cast("int")
        ).alias("first_name"),

        F.element_at(
            F.array(
                F.lit("Garcia"), F.lit("Rodriguez"),
                F.lit("Perez"), F.lit("Torres"),
                F.lit("Flores"), F.lit("Ramirez"),
                F.lit("Castillo"), F.lit("Vargas")
            ),
            (F.pmod("id", 8) + 1).cast("int")
        ).alias("last_name"),

        F.date_add(
            F.to_date(F.lit("1970-01-01")),
            (F.pmod("id", 12000) + 5000).cast("int")
        ).alias("birth_date"),

        F.when(F.pmod("id", 2) == 0, "M")
         .otherwise("F").alias("gender"),

        F.element_at(
            F.array(
                F.lit("Lima"), F.lit("Arequipa"),
                F.lit("Trujillo"), F.lit("Cusco"),
                F.lit("Piura"), F.lit("Chiclayo")
            ),
            (F.pmod("id", 6) + 1).cast("int")
        ).alias("city"),

        F.when(F.pmod("id", 3) == 0, "PREMIUM")
         .when(F.pmod("id", 3) == 1, "STANDARD")
         .otherwise("BASIC").alias("customer_segment"),

        F.date_add(
            F.to_date(F.lit("2020-01-01")),
            F.pmod("id", 2000).cast("int")
        ).alias("registration_date"),

        F.when(F.pmod("id", 20) == 0, "INACTIVE")
         .otherwise("ACTIVE").alias("customer_status"),

        F.current_timestamp().alias("ingestion_date")
    )
)

In [0]:
display(customers.limit(10))

In [0]:
# Validar customers
print("Total de clientes:", customers.count())
print("IDs distintos:", customers.select("customer_id").distinct().count())
print("Clientes sin identificador:", customers.filter(F.col("customer_id").isNull()).count())
print("Documentos duplicados:", customers.groupBy("document_number").count().filter(F.col("count") > 1).count())

In [0]:
#Escritura en contenedor raw
customers.write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .save(f"{base_path}/customers")

###Generar accounts

In [0]:
accounts = (
    spark.range(1, N_ACCOUNTS + 1)
    .select(
        F.col("id").cast("long").alias("account_id"),

        (
            F.pmod(F.col("id") - 1, F.lit(N_CUSTOMERS)) + 1
        ).cast("long").alias("customer_id"),

        F.when(F.pmod("id", 3) == 0, "SAVINGS")
         .when(F.pmod("id", 3) == 1, "CHECKING")
         .otherwise("PAYROLL").alias("account_type"),

        F.when(F.pmod("id", 4) == 0, "USD")
         .otherwise("PEN").alias("currency"),

        F.date_add(
            F.to_date(F.lit("2021-01-01")),
            F.pmod("id", 1800).cast("int")
        ).alias("opening_date"),

        F.round(
            (F.pmod("id", 100000) + 1000) / 100,
            2
        ).cast("decimal(18,2)").alias("balance"),

        F.when(F.pmod("id", 25) == 0, "INACTIVE")
         .otherwise("ACTIVE").alias("account_status"),

        F.current_timestamp().alias("ingestion_date")
    )
)

display(accounts.limit(10))

In [0]:
# Validación
print("Total de cuentas:", accounts.count())
print("IDs distintos:",accounts.select("account_id").distinct().count())
print("Cuentas sin cliente:",accounts.filter(F.col("customer_id").isNull()).count())
print("Clientes inexistentes:",accounts.join(customers.select("customer_id"),on="customer_id",how="left_anti").count())

In [0]:
# Escritura en contenedor raw
accounts.write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .save(f"{base_path}/accounts")

###Generar cards

In [0]:
cards = (
    spark.range(1, N_CARDS + 1)
    .select(
        F.col("id").cast("long").alias("card_id"),

        (
            F.pmod(F.col("id") - 1, F.lit(N_ACCOUNTS)) + 1
        ).cast("long").alias("account_id"),

        F.when(F.pmod("id", 2) == 0, "DEBIT")
         .otherwise("CREDIT").alias("card_type"),

        F.when(F.pmod("id", 3) == 0, "VISA")
         .otherwise("MASTERCARD").alias("card_brand"),

        F.date_add(
            F.to_date(F.lit("2022-01-01")),
            F.pmod("id", 1000).cast("int")
        ).alias("issue_date"),

        F.add_months(
            F.date_add(
                F.to_date(F.lit("2022-01-01")),
                F.pmod("id", 1000).cast("int")
            ),
            48
        ).alias("expiration_date"),

        F.when(F.pmod("id", 30) == 0, "BLOCKED")
         .otherwise("ACTIVE").alias("card_status"),

        F.current_timestamp().alias("ingestion_date")
    )
)

display(cards.limit(10))

In [0]:
# Validación
print("Total de tarjetas:", cards.count())
print("IDs distintos:",cards.select("card_id").distinct().count())
print("Cuentas inexistentes:",cards.join(accounts.select("account_id"),on="account_id",how="left_anti").count())

In [0]:
# Escritura en contenedor raw
cards.write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .save(f"{base_path}/cards")

###Generar transactions

In [0]:
transactions = (
    spark.range(1, N_TRANSACTIONS + 1)
    .select(
        F.col("id").cast("long").alias("transaction_id"),

        (
            F.pmod(F.col("id") - 1, F.lit(N_ACCOUNTS)) + 1
        ).cast("long").alias("account_id"),

        F.when(
            F.pmod("id", 5) != 0,
            (
                F.pmod(F.col("id") - 1, F.lit(N_CARDS)) + 1
            ).cast("long")
        ).otherwise(F.lit(None).cast("long")).alias("card_id"),

        F.expr("""
            timestampadd(
                SECOND,
                CAST(id * 120 AS INT),
                TIMESTAMP '2025-01-01 00:00:00'
            )
        """).alias("transaction_date"),

        F.when(F.pmod("id", 4) == 0, "PURCHASE")
         .when(F.pmod("id", 4) == 1, "TRANSFER")
         .when(F.pmod("id", 4) == 2, "WITHDRAWAL")
         .otherwise("DEPOSIT").alias("transaction_type"),

        F.when(F.pmod("id", 3) == 0, "ONLINE")
         .when(F.pmod("id", 3) == 1, "POS")
         .otherwise("ATM").alias("channel"),

        F.when(F.pmod("id", 5) == 0, "FOOD")
         .when(F.pmod("id", 5) == 1, "RETAIL")
         .when(F.pmod("id", 5) == 2, "TRAVEL")
         .when(F.pmod("id", 5) == 3, "ENTERTAINMENT")
         .otherwise("OTHER").alias("merchant_category"),

        F.round(
            (F.pmod("id", 50000) + 100) / 100,
            2
        ).cast("decimal(18,2)").alias("amount"),

        F.when(F.pmod("id", 10) == 0, "USD")
         .otherwise("PEN").alias("currency"),

        F.when(F.pmod("id", 50) == 0, "REJECTED")
         .otherwise("APPROVED").alias("transaction_status"),

        F.when(F.pmod("id", 200) == 0, 1)
         .otherwise(0).alias("is_fraud"),

        F.current_timestamp().alias("ingestion_date")
    )
)

display(transactions.limit(10))

In [0]:
# Validación
print("Total de transacciones:", transactions.count())
print("IDs distintos:",transactions.select("transaction_id").distinct().count())
print("Cuentas inexistentes:",transactions.join(accounts.select("account_id"),on="account_id",how="left_anti").count())
print("Tarjetas inexistentes:",transactions.filter(F.col("card_id").isNotNull()).join(cards.select("card_id"),on="card_id",how="left_anti").count())

display(transactions.groupBy("is_fraud").count().orderBy("is_fraud"))

In [0]:
# Validar y relacionar transactions - cards
transactions = (
    transactions
    .withColumn(
        "card_id",
        F.when(
            (F.col("account_id") <= N_CARDS) &
            (F.pmod(F.col("transaction_id"), 5) != 0),
            F.col("account_id")
        ).otherwise(F.lit(None).cast("long"))
    )
)

# Validar que la tarjeta pertenezca a la cuenta
invalid_card_account = (
    transactions.alias("t")
    .filter(F.col("t.card_id").isNotNull())
    .join(
        cards.select("card_id", "account_id").alias("c"),
        F.col("t.card_id") == F.col("c.card_id"),
        "inner"
    )
    .filter(F.col("t.account_id") != F.col("c.account_id"))
    .count()
)

print("Relaciones tarjeta-cuenta incorrectas:", invalid_card_account)

In [0]:
# Escritura en contenedor raw
transactions.write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .save(f"{base_path}/transactions")

###Generar loans

In [0]:
loans = (
    spark.range(1, N_LOANS + 1)
    .select(
        F.col("id").cast("long").alias("loan_id"),

        (
            F.pmod(F.col("id") - 1, F.lit(N_CUSTOMERS)) + 1
        ).cast("long").alias("customer_id"),

        F.when(F.pmod("id", 3) == 0, "PERSONAL")
         .when(F.pmod("id", 3) == 1, "VEHICLE")
         .otherwise("CONSUMER").alias("loan_type"),

        F.round(
            (F.pmod("id", 100000) + 100000) / 100,
            2
        ).cast("decimal(18,2)").alias("principal_amount"),

        F.round(
            (F.pmod("id", 1200) + 500) / 100,
            4
        ).cast("decimal(8,4)").alias("interest_rate"),

        F.when(F.pmod("id", 3) == 0, 12)
         .when(F.pmod("id", 3) == 1, 24)
         .otherwise(36).alias("term_months"),

        F.date_add(
            F.to_date(F.lit("2024-01-01")),
            F.pmod("id", 700).cast("int")
        ).alias("disbursement_date"),

        F.add_months(
            F.date_add(
                F.to_date(F.lit("2024-01-01")),
                F.pmod("id", 700).cast("int")
            ),
            F.when(F.pmod("id", 3) == 0, 12)
             .when(F.pmod("id", 3) == 1, 24)
             .otherwise(36)
        ).alias("due_date"),

        F.when(F.pmod("id", 20) == 0, "PAID")
         .otherwise("ACTIVE").alias("loan_status"),

        F.current_timestamp().alias("ingestion_date")
    )
)

display(loans.limit(10))

In [0]:
# Validación
print("Total de préstamos:", loans.count())
print("Clientes inexistentes:",loans.join(customers.select("customer_id"),on="customer_id",how="left_anti").count())

In [0]:
# Escritura en contenedor raw
loans.write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .save(f"{base_path}/loans")

###Generar payments

In [0]:
payments = (
    spark.range(1, N_PAYMENTS + 1)
    .select(
        F.col("id").cast("long").alias("payment_id"),

        (
            F.pmod(F.col("id") - 1, F.lit(N_LOANS)) + 1
        ).cast("long").alias("loan_id"),

        F.date_add(
            F.to_date(F.lit("2025-01-01")),
            F.pmod("id", 365).cast("int")
        ).alias("payment_date"),

        F.round(
            (F.pmod("id", 50000) + 10000) / 100,
            2
        ).cast("decimal(18,2)").alias("payment_amount"),

        F.round(
            (F.pmod("id", 35000) + 5000) / 100,
            2
        ).cast("decimal(18,2)").alias("principal_paid"),

        F.round(
            (F.pmod("id", 10000) + 1000) / 100,
            2
        ).cast("decimal(18,2)").alias("interest_paid"),

        F.when(F.pmod("id", 20) == 0, "PENDING")
         .otherwise("PAID").alias("payment_status"),

        F.current_timestamp().alias("ingestion_date")
    )
)

display(payments.limit(10))

In [0]:
# Validación
print("Total de pagos:", payments.count())
print("Préstamos inexistentes:",payments.join(loans.select("loan_id"),on="loan_id",how="left_anti").count())

In [0]:
# Escritura en contenedor raw
payments.write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .save(f"{base_path}/payments")

In [0]:
# Validar los datasets generados
datasets = {
    "customers": customers,
    "accounts": accounts,
    "cards": cards,
    "transactions": transactions,
    "loans": loans,
    "payments": payments
}

for name, df in datasets.items():

    print(f"{name:<15} | {df.count():>10,} registros")